# MNIST Handwritten Digit Classification with a Neural Network


## 1. Environment Setup


In [ ]:
# Colab: run this cell first. It clones/checks out the JWH branch and loads src.
# The GitHub URL and branch are fixed, so you do not need to type them.
import os
import sys
from pathlib import Path

GIT_URL = "https://github.com/Jungle-12-303/wk13_team2_mnist.git"
BRANCH = "JWH"
REPO_NAME = "wk13_team2_mnist"

if "google.colab" in sys.modules:
    repo_path = Path("/content") / REPO_NAME

    # If the repo already exists, reuse it and pull the latest JWH branch.
    if repo_path.exists():
        os.chdir(repo_path)
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    else:
        os.chdir("/content")
        !git clone -b {BRANCH} {GIT_URL}
        os.chdir(repo_path)

    src_path = str(Path.cwd() / "src")
    if src_path in sys.path:
        sys.path.remove(src_path)
    sys.path.insert(0, src_path)
else:
    src_path = str(Path.cwd() / "src")
    if src_path in sys.path:
        sys.path.remove(src_path)
    sys.path.insert(0, src_path)

# Clear cached modules so Colab does not keep old code from another branch.
for module_name in ["training", "network", "losses", "layers", "activations", "optimizers", "data"]:
    sys.modules.pop(module_name, None)

print("Current working directory:", Path.cwd())
print("Current branch:")
!git branch --show-current


## 2. Load Data


In [ ]:
from data import load_mnist

(x_train, y_train), (x_test, y_test) = load_mnist()
print('Train:', x_train.shape, y_train.shape)
print('Test:', x_test.shape, y_test.shape)

## 3. Run Tests

Run the test cell below when you want to verify the implementation.
- Main implementation files: `activations.py`, `layers.py`, `losses.py`, `optimizers.py`, `network.py`, `training.py`
- Example imports: `from activations import ReLU`, `from network import NeuralNetwork`
- Start with one test file, then run all tests when ready.
    - ReLU only: `TEST_TARGET = "tests/test_relu.py"`
    - Filter tests by keyword: `PYTEST_KEYWORD = "backward"`
    - All tests: `TEST_TARGET = "tests/"`


In [ ]:
import subprocess
import sys
from pathlib import Path

# Use the current notebook directory as the repository root.
repo_dir = Path.cwd()

# Start with the test file you are working on.
# Examples: tests/test_relu.py, tests/test_affine.py, tests/test_training.py
TEST_TARGET = "tests/test_relu.py"

# Use this to run only tests whose names contain a keyword.
# Example: "backward". Leave empty to run the whole file.
PYTEST_KEYWORD = ""

cmd = [sys.executable, "-m", "pytest", TEST_TARGET, "-v"]
if PYTEST_KEYWORD:
    cmd.extend(["-k", PYTEST_KEYWORD])

print("Working directory:", repo_dir)
print("Command:", " ".join(cmd))
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    cwd=str(repo_dir)
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode == 0:
    print("\nSelected tests passed.")
else:
    print("\nSome selected tests failed.")


## 4. Create Model and Train


In [ ]:
from network import NeuralNetwork
from optimizers import Adam
from training import train

model = NeuralNetwork(use_batchnorm=True, use_dropout=True)
optimizer = Adam(lr=0.001)

loss_history = train(model, optimizer, x_train, y_train, epochs=20, batch_size=128)


## 5. Evaluate and Plot Loss


In [ ]:
from training import evaluate, plot_loss_history

acc, n_params = evaluate(model, x_test, y_test)
print(f'Test Accuracy: {acc:.2f}%')
print(f'Total Params: {n_params:,}')

plot_loss_history(loss_history)

## JWH Branch Setup

Run this cell before running the experiment runner.
It updates the notebook to the latest `JWH` branch and clears import caches.


In [ ]:
import os
import sys
from pathlib import Path

REPO_NAME = "wk13_team2_mnist"
repo_dir = Path.cwd()
if not (repo_dir / "src").exists() and Path(f"/content/{REPO_NAME}/src").exists():
    repo_dir = Path(f"/content/{REPO_NAME}")
os.chdir(repo_dir)

!git fetch origin
!git checkout JWH
!git pull origin JWH

src_path = str(Path.cwd() / "src")
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

for module_name in ["training", "network", "losses", "layers", "activations", "optimizers", "data"]:
    sys.modules.pop(module_name, None)

print("Ready on branch JWH")
!git branch --show-current


## Experiment Runner

This cell runs one or more MLP experiment configs and saves `results.csv` and `results.json`.
Start with `configs[:1]` for a quick baseline run. Use all configs when you are ready.


In [ ]:
from data import load_mnist
from training import DEFAULT_EXPERIMENT_CONFIGS, run_experiments, plot_compare_loss_histories

(x_train, y_train), (x_test, y_test) = load_mnist()

# Quick check: run only the baseline experiment.
configs = DEFAULT_EXPERIMENT_CONFIGS[:1]

# Full comparison: uncomment the next line.
# configs = DEFAULT_EXPERIMENT_CONFIGS

results, histories = run_experiments(
    configs,
    x_train, y_train,
    x_test, y_test,
    csv_path="results.csv",
    json_path="results.json",
)

results


## 10. Multi Experiment Suite

Run these cells independently after loading the data. They build several models by combining the existing components with the added Sigmoid activation, then record the required experiment fields: model structure, training settings, BatchNorm/Dropout, initialization, final train loss, test accuracy, parameter count, and elapsed time.

In [ ]:
# Multi-experiment configuration cell
# Uses the required loop implemented in training.train():
# Forward(train=True) -> cross_entropy_loss -> backward -> optimizer.update.

from copy import deepcopy
from data import load_mnist
from training import run_experiments, summarize_experiment_results

(x_train, y_train), (x_test, y_test) = load_mnist()

BASE_CONFIG = {
    "hidden_dims": [256, 128],
    "activation": "relu",
    "use_batchnorm": True,
    "dropout_rate": 0.2,
    "weight_init": "he",
    "optimizer": "adam",
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 128,
    "seed": 42,
}

def make_config(name, **overrides):
    config = deepcopy(BASE_CONFIG)
    config.update(overrides)
    config["name"] = name
    return config

# Original components + Sigmoid combinations.
ARCHITECTURE_CONFIGS = [
    make_config("relu_256_128", hidden_dims=[256, 128], activation="relu", weight_init="he"),
    make_config("relu_512_256_128", hidden_dims=[512, 256, 128], activation="relu", weight_init="he"),
    make_config("relu_1024_512", hidden_dims=[1024, 512], activation="relu", weight_init="he"),
    make_config("sigmoid_256_128", hidden_dims=[256, 128], activation="sigmoid", weight_init="xavier"),
    make_config("sigmoid_512_256", hidden_dims=[512, 256], activation="sigmoid", weight_init="xavier"),
]

DROPOUT_CONFIGS = [
    make_config("dropout_0_0", dropout_rate=0.0),
    make_config("dropout_0_1", dropout_rate=0.1),
    make_config("dropout_0_2", dropout_rate=0.2),
    make_config("dropout_0_5", dropout_rate=0.5),
]

LR_CONFIGS = [
    make_config("lr_0_0005", lr=0.0005),
    make_config("lr_0_001", lr=0.001),
    make_config("lr_0_002", lr=0.002),
    make_config("lr_decay_10_15", lr=0.001, lr_decay_epochs=[10, 15], lr_decay_factor=0.1),
]

BATCHNORM_CONFIGS = [
    make_config("batchnorm_on", use_batchnorm=True),
    make_config("batchnorm_off", use_batchnorm=False),
    make_config("sigmoid_batchnorm_on", activation="sigmoid", weight_init="xavier", use_batchnorm=True),
    make_config("sigmoid_batchnorm_off", activation="sigmoid", weight_init="xavier", use_batchnorm=False),
]

QUICK_CONFIGS = [
    make_config("quick_relu", epochs=3),
    make_config("quick_sigmoid", activation="sigmoid", weight_init="xavier", epochs=3),
]

EXPERIMENT_GROUPS = {
    "quick": QUICK_CONFIGS,
    "architecture": ARCHITECTURE_CONFIGS,
    "dropout": DROPOUT_CONFIGS,
    "learning_rate": LR_CONFIGS,
    "batchnorm": BATCHNORM_CONFIGS,
    "full": ARCHITECTURE_CONFIGS + DROPOUT_CONFIGS + LR_CONFIGS + BATCHNORM_CONFIGS,
}

# Change this value to run another group: quick, architecture, dropout, learning_rate, batchnorm, full.
SELECTED_GROUP = "quick"
configs = EXPERIMENT_GROUPS[SELECTED_GROUP]
print(f"Selected group: {SELECTED_GROUP} ({len(configs)} experiments)")
[name for name in EXPERIMENT_GROUPS]

In [ ]:
# Run selected experiments.
# Elapsed time is measured inside run_experiments/run_experiment and stored as time_sec.

results, histories = run_experiments(
    configs,
    x_train, y_train,
    x_test, y_test,
    csv_path=f"results_{SELECTED_GROUP}.csv",
    json_path=f"results_{SELECTED_GROUP}.json",
)

summarize_experiment_results(results)

In [ ]:
# Pretty result table without requiring pandas.
from IPython.display import HTML, display

def format_result_table(results):
    headers = [
        "name", "architecture", "activation", "optimizer", "lr", "epochs",
        "batch_size", "batchnorm", "dropout", "init", "train_loss",
        "test_acc", "params", "time_sec",
    ]
    rows = sorted(results, key=lambda row: row["test_acc"], reverse=True)

    def fmt(value, key):
        if key in {"train_loss"}:
            return "" if value is None else f"{value:.4f}"
        if key in {"test_acc"}:
            return f"{value:.2f}%"
        if key == "time_sec":
            return f"{value:.1f}s"
        if key == "params":
            return f"{value:,}"
        return str(value)

    html = [
        "<style>",
        ".exp-table {border-collapse: collapse; font-size: 13px; width: 100%;}",
        ".exp-table th, .exp-table td {border: 1px solid #ddd; padding: 6px 8px; text-align: right;}",
        ".exp-table th {background: #f4f4f4;}",
        ".exp-table td:first-child, .exp-table th:first-child {text-align: left;}",
        "</style>",
        "<table class='exp-table'><thead><tr>",
    ]
    html.extend(f"<th>{header}</th>" for header in headers)
    html.append("</tr></thead><tbody>")
    for row in rows:
        html.append("<tr>")
        html.extend(f"<td>{fmt(row.get(header), header)}</td>" for header in headers)
        html.append("</tr>")
    html.append("</tbody></table>")
    return "".join(html)

display(HTML(format_result_table(results)))

In [ ]:
# Visualization cell: loss curves, accuracy comparison, LR comparison, and Dropout comparison.
from pathlib import Path
import matplotlib.pyplot as plt

figure_dir = Path("figures")
figure_dir.mkdir(exist_ok=True)

def save_and_show(name):
    path = figure_dir / f"{SELECTED_GROUP}_{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"saved: {path}")

# 1) Epoch별 loss curve
plt.figure(figsize=(9, 5))
for result in results:
    plt.plot(result["loss_history"], marker="o", linewidth=2, label=result["name"])
plt.xlabel("Epoch")
plt.ylabel("Train Loss")
plt.title(f"Loss Curves - {SELECTED_GROUP}")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
save_and_show("loss_curves")

# 2) Test accuracy comparison
ordered = sorted(results, key=lambda row: row["test_acc"], reverse=True)
plt.figure(figsize=(10, 5))
plt.bar([row["name"] for row in ordered], [row["test_acc"] for row in ordered])
plt.axhline(95, color="tab:orange", linestyle="--", linewidth=1.5, label="95% target")
plt.axhline(97, color="tab:green", linestyle="--", linewidth=1.5, label="97% recommended")
plt.ylabel("Test Accuracy (%)")
plt.title(f"Test Accuracy - {SELECTED_GROUP}")
plt.xticks(rotation=30, ha="right")
plt.ylim(max(0, min(row["test_acc"] for row in ordered) - 2), 100)
plt.grid(axis="y", alpha=0.25)
plt.legend()
save_and_show("test_accuracy")

# 3) Learning rate comparison if the selected results include LR experiments
lr_rows = [row for row in results if row["name"].startswith("lr_")]
if lr_rows:
    plt.figure(figsize=(7, 4))
    plt.plot([str(row["lr"]) for row in lr_rows], [row["test_acc"] for row in lr_rows], marker="o", linewidth=2)
    plt.xlabel("Learning Rate")
    plt.ylabel("Test Accuracy (%)")
    plt.title("Learning Rate vs Accuracy")
    plt.grid(True, alpha=0.3)
    save_and_show("learning_rate_accuracy")

# 4) Dropout before/after comparison if the selected results include Dropout experiments
dropout_rows = [row for row in results if row["name"].startswith("dropout_")]
if dropout_rows:
    plt.figure(figsize=(7, 4))
    plt.plot([row["dropout"] for row in dropout_rows], [row["test_acc"] for row in dropout_rows], marker="o", linewidth=2)
    plt.xlabel("Dropout Rate")
    plt.ylabel("Test Accuracy (%)")
    plt.title("Dropout Rate vs Accuracy")
    plt.grid(True, alpha=0.3)
    save_and_show("dropout_accuracy")

## 11. Lightweight Model Search

This experiment checks how far the MLP can be reduced while still meeting the original assignment accuracy target. The main selection rule is:

- Recommended target: choose the smallest model with `test_acc >= 97.0%`.
- Minimum assignment target: also report the smallest model with `test_acc >= 95.0%` as a fallback.

All candidates use the same NumPy training loop as the main experiments, so the comparison is based on parameter count, test accuracy, and training time rather than a different implementation.


In [ ]:
# Lightweight model search
# Goal: find the smallest model that still satisfies the original accuracy standard.
# Recommended standard in the report: >= 97% test accuracy.
# Minimum assignment standard: >= 95% test accuracy.

from copy import deepcopy
from pathlib import Path
import matplotlib.pyplot as plt

from data import load_mnist
from training import run_experiments, summarize_experiment_results

(x_train, y_train), (x_test, y_test) = load_mnist()

LIGHTWEIGHT_BASE_CONFIG = {
    "hidden_dims": [64, 32],
    "activation": "relu",
    "use_batchnorm": False,
    "dropout_rate": 0.0,
    "weight_init": "he",
    "optimizer": "adam",
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 128,
    "seed": 42,
}


def make_light_config(name, **overrides):
    config = deepcopy(LIGHTWEIGHT_BASE_CONFIG)
    config.update(overrides)
    config["name"] = name
    return config


# Start from the previously good practical model, then shrink width step by step.
# BatchNorm/dropout are off by default here because earlier experiments showed
# batchnorm_off was faster and used fewer parameters while staying accurate.
LIGHTWEIGHT_CONFIGS = [
    make_light_config("tiny_256_128_reference", hidden_dims=[256, 128]),
    make_light_config("tiny_128_64", hidden_dims=[128, 64]),
    make_light_config("tiny_96_48", hidden_dims=[96, 48]),
    make_light_config("tiny_64_32", hidden_dims=[64, 32]),
    make_light_config("tiny_48_24", hidden_dims=[48, 24]),
    make_light_config("tiny_32_16", hidden_dims=[32, 16]),
    make_light_config("tiny_32", hidden_dims=[32]),
    make_light_config("tiny_16", hidden_dims=[16]),
]

# Optional quick smoke test before the full run. Set to True to run only 3 epochs.
RUN_LIGHTWEIGHT_SMOKE_TEST = False
if RUN_LIGHTWEIGHT_SMOKE_TEST:
    lightweight_configs = [
        {**config, "name": f"smoke_{config['name']}", "epochs": 3}
        for config in LIGHTWEIGHT_CONFIGS[1:4]
    ]
    lightweight_group_name = "lightweight_smoke"
else:
    lightweight_configs = LIGHTWEIGHT_CONFIGS
    lightweight_group_name = "lightweight_search"

lightweight_results, lightweight_histories = run_experiments(
    lightweight_configs,
    x_train, y_train,
    x_test, y_test,
    csv_path=f"results_{lightweight_group_name}.csv",
    json_path=f"results_{lightweight_group_name}.json",
)

summarize_experiment_results(lightweight_results)


def pick_smallest_above(results, min_acc):
    passed = [row for row in results if row["test_acc"] >= min_acc]
    if not passed:
        return None
    return min(passed, key=lambda row: (row["params"], -row["test_acc"], row["time_sec"]))


recommended_pick = pick_smallest_above(lightweight_results, 97.0)
minimum_pick = pick_smallest_above(lightweight_results, 95.0)

for label, pick in [("Smallest model >= 97%", recommended_pick), ("Smallest model >= 95%", minimum_pick)]:
    print(f"\n{label}")
    if pick is None:
        print("No candidate reached this target. Try a wider model or more epochs.")
    else:
        print(
            f"name={pick['name']}, architecture={pick['architecture']}, "
            f"test_acc={pick['test_acc']:.2f}%, params={pick['params']:,}, "
            f"time={pick['time_sec']:.1f}s"
        )

# Plot accuracy vs model size. The best lightweight choice should be far left
# while staying above the target line.
figure_dir = Path("figures")
figure_dir.mkdir(exist_ok=True)
ordered = sorted(lightweight_results, key=lambda row: row["params"])

plt.figure(figsize=(9, 5))
plt.plot([row["params"] for row in ordered], [row["test_acc"] for row in ordered], marker="o", linewidth=2)
for row in ordered:
    plt.annotate(row["name"].replace("tiny_", ""), (row["params"], row["test_acc"]), fontsize=8, xytext=(5, 4), textcoords="offset points")
plt.axhline(97, color="tab:green", linestyle="--", linewidth=1.5, label="97% recommended")
plt.axhline(95, color="tab:orange", linestyle="--", linewidth=1.5, label="95% minimum")
plt.xlabel("Trainable Parameters")
plt.ylabel("Test Accuracy (%)")
plt.title("Lightweight Model Search: Params vs Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plot_path = figure_dir / f"{lightweight_group_name}_params_vs_accuracy.png"
plt.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"saved: {plot_path}")

# Compact table sorted by parameter count.
try:
    from IPython.display import HTML, display

    rows = []
    for row in ordered:
        rows.append(
            "<tr>"
            f"<td>{row['name']}</td>"
            f"<td>{row['architecture']}</td>"
            f"<td>{row['test_acc']:.2f}%</td>"
            f"<td>{row['params']:,}</td>"
            f"<td>{row['time_sec']:.1f}s</td>"
            "</tr>"
        )
    display(HTML(
        "<table style='border-collapse:collapse;font-size:13px;width:100%'>"
        "<thead><tr>"
        "<th style='border:1px solid #ddd;padding:6px;text-align:left'>name</th>"
        "<th style='border:1px solid #ddd;padding:6px;text-align:left'>architecture</th>"
        "<th style='border:1px solid #ddd;padding:6px;text-align:right'>test_acc</th>"
        "<th style='border:1px solid #ddd;padding:6px;text-align:right'>params</th>"
        "<th style='border:1px solid #ddd;padding:6px;text-align:right'>time</th>"
        "</tr></thead><tbody>"
        + "".join(rows)
        + "</tbody></table>"
    ))
except Exception:
    pass

lightweight_results
